# 🔥 FlameMind SLM — Huấn luyện Phán quyết cháy Đa phương thức trên Kaggle

> **Đề tài NCKH:** *Nghiên cứu và phát triển hệ thống cảnh báo cháy sử dụng mô hình đa phương thức (Multimodal Model)*  
> **Mục tiêu Notebook:** Fine-tune mô hình ngôn ngữ nhỏ **Qwen2.5-1.5B-Instruct** bằng kỹ thuật **QLoRA 4-bit (Unsloth)** trên toàn bộ **20.026 kịch bản** (20.000 kịch bản động học cháy tổng hợp + kịch bản thực tế từ MmodalFire).

### ⚡ Lợi thế khi chạy trên Kaggle GPU (T4 x2 hoặc P100 16GB VRAM):
- Bộ nhớ VRAM lớn (16GB) giúp tăng batch size và tận dụng flash-attention.
- Thời gian huấn luyện ước tính: chỉ khoảng **1.5 – 2 giờ** (so với ~10 giờ trên laptop).
- Tự động đóng gói kết quả thành file `.zip` để tải về máy sau khi huấn luyện xong.

## 1. Kiểm tra GPU Kaggle
*(Hãy đảm bảo bạn đã bật GPU trong thanh cấu hình bên phải: **Settings -> Accelerator -> GPU T4 x2 hoặc P100** và bật **Internet = ON**)*

In [ ]:
!nvidia-smi

## 2. Cài đặt Unsloth và các thư viện cần thiết

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers triton

## 3. Sinh dữ liệu kịch bản Động học cháy (20.000 mẫu)
*(Tự động sinh trực tiếp trên môi trường Kaggle, hoàn toàn độc lập và không phụ thuộc vào kết nối mạng ngoài)*

In [ ]:
import os
import json
import random
from pathlib import Path

FAMILIES = [
    ("smolder_burst", "chay_that", 25.0, 75.0, 140.0, 9.0),
    ("fast_flame", "chay_that", 18.0, 90.0, 90.0, 14.0),
    ("electrical", "chay_that", 12.0, 65.0, 60.0, 7.0),
    ("candle_benign", "lanh_tinh", 1.5, 31.0, 8.0, 0.3),
    ("cooking_steam", "lanh_tinh", 4.0, 34.0, 12.0, 0.6),
    ("incense", "lanh_tinh", 2.5, 29.0, 9.0, 0.2),
    ("dust_burst", "khong_chay", 6.0, 27.0, 4.0, 0.1),
    ("fog_morning", "khong_chay", 5.0, 26.0, 3.0, 0.1),
]

ROOMS = ["bep", "phong_khach", "hanh_lang", "kho", "ban_tho", "nha_xe"]

def sensor_curve(peak: float, tau_s: float, rng: random.Random, seconds: int = 60) -> list:
    return [round(peak * (1 - 2.71828 ** (-t / tau_s)) + rng.gauss(0, peak * 0.02), 3) for t in range(seconds)]

def make_scenario(family: tuple, rng: random.Random) -> dict:
    name, verdict, smoke, temp, co, ror = family
    return {
        "family": name,
        "verdict": verdict,
        "room": rng.choice(ROOMS),
        "hour": rng.randint(0, 23),
        "sensors": {
            "smoke_obs_per_m": sensor_curve(smoke, rng.uniform(8, 25), rng),
            "temperature_c": sensor_curve(temp - 26, rng.uniform(20, 45), rng),
            "co_ppm": sensor_curve(co, rng.uniform(15, 40), rng),
            "rate_of_rise_c_per_min": round(ror * rng.uniform(0.8, 1.2), 2),
        },
        "cot_template": [
            "kiem_tra_tinh_nhat_quan_da_phuong_thuc",
            "kiem_tra_dong_hoc_tang_dan",
            "kiem_tra_nguon_lanh_tinh_da_biet",
            "ket_luan_va_muc_canh_bao",
        ],
    }

# Đảm bảo thư mục đích tồn tại
out_dir = Path("FlameMind/data/processed/scenarios")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "scenes_sft.jsonl"

print("⏳ Đang sinh 20.000 kịch bản mô phỏng động học cháy...")
rng = random.Random(42)
with open(out_file, "w", encoding="utf-8") as fh:
    for _ in range(20000):
        fh.write(json.dumps(make_scenario(rng.choice(FAMILIES), rng), ensure_ascii=False) + "\n")

# Lưu thêm một bản tại thư mục hiện tại để tiện tra cứu
if not os.path.exists("scenes_sft.jsonl"):
    os.system(f"cp {out_file} scenes_sft.jsonl")

print(f"✅ Đã tạo thành công file kịch bản tại: {out_file} ({os.path.getsize(out_file) / (1024*1024):.2f} MB)")

## 4. Tải mô hình nền Qwen2.5-1.5B và Cấu hình QLoRA (Unsloth)

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024
base_model = "Qwen/Qwen2.5-1.5B-Instruct"

# Tải mô hình nén 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# Cấu hình tham số LoRA (1.18% trainable parameters)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

print("✅ Mô hình Qwen2.5-1.5B LoRA đã sẵn sàng!")

## 5. Chuẩn bị tập dữ liệu huấn luyện (Định dạng Chat Template)

In [ ]:
import json
from datasets import Dataset

PROMPT = (
    "Bạn là model phán quyết cháy chạy trong tòa nhà. Gói ngữ cảnh: {ctx}\n"
    "Trả lời đúng một JSON gồm verdict thuộc chay_that|lanh_tinh|khong_chay, confidence, chuoi_suy_luan."
)

def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

def to_messages(r):
    ctx = json.dumps({"room": r.get("room"), "hour": r.get("hour"), "sensors": r.get("sensors")}, ensure_ascii=False)
    ans = json.dumps({"verdict": r["verdict"], "confidence": 0.9,
                      "chuoi_suy_luan": r.get("cot_template", [])}, ensure_ascii=False)
    return [
        {"role": "user", "content": PROMPT.format(ctx=ctx)},
        {"role": "assistant", "content": ans},
    ]

# Kiểm tra các đường dẫn có thể có của file kịch bản
scenario_candidates = [
    "FlameMind/data/processed/scenarios/scenes_sft.jsonl",
    "scenes_sft.jsonl",
    "data/processed/scenarios/scenes_sft.jsonl"
]

chosen_scenario_file = None
for path in scenario_candidates:
    if os.path.exists(path):
        chosen_scenario_file = path
        break

if not chosen_scenario_file:
    # Tự động sinh ngay tại chỗ nếu chưa có
    print("⚠️ Chưa thấy file kịch bản, đang tự động sinh 20.000 kịch bản...")
    os.makedirs("FlameMind/data/processed/scenarios", exist_ok=True)
    chosen_scenario_file = "FlameMind/data/processed/scenarios/scenes_sft.jsonl"
    rng = random.Random(42)
    with open(chosen_scenario_file, "w", encoding="utf-8") as fh:
        for _ in range(20000):
            fh.write(json.dumps(make_scenario(rng.choice(FAMILIES), rng), ensure_ascii=False) + "\n")

rows = load_jsonl(chosen_scenario_file)

# Nạp thêm MmodalFire train nếu có
for mmodal_path in ["FlameMind/data/processed/verdict/mmodalfire_train.jsonl", "data/processed/verdict/mmodalfire_train.jsonl"]:
    if os.path.exists(mmodal_path):
        m_rows = load_jsonl(mmodal_path)
        rows += m_rows
        print(f"➕ Đã bổ sung {len(m_rows)} kịch bản từ MmodalFire")
        break

print(f"📊 Tổng số mẫu đưa vào huấn luyện: {len(rows)} mẫu")

# Chuyển đổi sang chat template
texts = [tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False) for r in rows]
train_ds = Dataset.from_list([{"text": t} for t in texts])
print("✅ Hoàn tất format tập dữ liệu SFT!")

## 6. Huấn luyện SFT với SFTTrainer (2 Epochs)
*(Trên Kaggle VRAM 16GB, ta sử dụng batch_size = 4, gradient_accumulation = 2 để tối ưu tốc độ)*

In [ ]:
from trl import SFTConfig, SFTTrainer

output_dir = "output_models/slm-0.1.0"

cfg = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=4,        # Tăng lên 4 nhờ 16GB VRAM
    gradient_accumulation_steps=2,        # Total batch size hiệu dụng = 8
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="epoch",
    seed=42,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=cfg,
)

print("🚀 Bắt đầu quá trình huấn luyện...")
trainer_stats = trainer.train()
print("🎉 Huấn luyện thành công!")

## 7. Lưu Adapter và Đóng gói file ZIP tải về máy

In [ ]:
save_adapter_dir = "flamemind_slm_0.1.0_adapter"

# Lưu trọng số adapter và tokenizer
model.save_pretrained(save_adapter_dir)
tokenizer.save_pretrained(save_adapter_dir)
print(f"💾 Đã lưu adapter tại: {save_adapter_dir}")

# Nén file zip để tải về trực tiếp từ tab Output của Kaggle
!zip -r flamemind_slm_0.1.0_adapter.zip {save_adapter_dir}
print("📦 File nén đã sẵn sàng: /kaggle/working/flamemind_slm_0.1.0_adapter.zip")

## 8. Chạy thử nghiệm suy luận nhanh (Test Inference)

In [ ]:
FastLanguageModel.for_inference(model)

# Trường hợp giả lập: cháy thật tại kho lúc 2h đêm
test_scenario = {
    "room": "kho",
    "hour": 2,
    "sensors": {
        "smoke_obs_per_m": [0.2, 0.8, 3.5, 9.2, 17.5, 23.4],
        "temperature_c": [28.0, 31.2, 39.5, 56.0, 74.8],
        "co_ppm": [4, 15, 42, 85, 135],
        "rate_of_rise_c_per_min": 12.4
    }
}

test_ctx = json.dumps(test_scenario, ensure_ascii=False)
inputs = tokenizer.apply_chat_template([
    {"role": "user", "content": PROMPT.format(ctx=test_ctx)}
], tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
pred_text = tokenizer.batch_decode(outputs)[0]

print("=== KẾT QUẢ PHÁN QUYẾT TỪ SLM ===")
print(pred_text.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip())